In [1]:
import logging
import os
import nibabel as nib
import re
from nilearn.image import index_img, load_img
from joblib import Parallel, delayed

In [2]:

log_dir = "/scratch/nbe/prevent/Baran/Codes/Trimming_logs"
os.makedirs(log_dir, exist_ok=True)

log_file = os.path.join(log_dir, "nifti_trimming.log")

# kill old handlers so logs don't repeat in Jupyter
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

# log config - overwrite file each run
logging.basicConfig(
    filename=log_file,
    filemode='w',  # write mode, so it doesn't stack logs
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
     datefmt="%H:%M:%S", # show only time
)

# also show logs in terminal
console = logging.StreamHandler()
console.setLevel(logging.INFO)
formatter = logging.Formatter("%(asctime)s [%(levelname)s] %(message)s")
console.setFormatter(formatter)
logging.getLogger().addHandler(console)


In [3]:
def nifti_trimmer(task, subId):
    
    logging.info(f"▶️ Trimming {subId}")    

    
    base_trimmed_dir = "/scratch/nbe/prevent/Baran/Codes/trimmed_nifti"
    subject_trimmed_dir = os.path.join(base_trimmed_dir, subId)
    
    # Create the subject-specific output folder if it doesn't exist
    os.makedirs(subject_trimmed_dir, exist_ok=True)
    
    paths = [nifti_finder(task, subId, run) for run in range(1, 6)]
    trimmed_paths = []

    lengths = [extract_length_from_log(task, subId, run) for run in range(1, 6)]
    
    for i, (path, length) in enumerate(zip(paths, lengths), start=1):

        img = load_img(path)
        trimmed_img = index_img(img, slice(length))
        
        out_path = os.path.join(subject_trimmed_dir, f"trimmed_bold_run{i}.nii.gz")
        trimmed_img.to_filename(out_path)
        trimmed_paths.append(out_path)

        logging.info(f"✔️ Trimmed run {i} for {subId}, length={length}, output saved to {out_path}")
    return trimmed_paths

In [11]:
def extract_length_from_log(task, subId, run):
    

    logging.info(f"🔮 extracting frame count for {subId}, run {run}")
    
    log_path = f"/scratch/nbe/prevent/Experiment1/Data_BIDS/derivatives/keras/{subId}/logs/virtual_contact_sce{run}.log"
    
    expected_columns = ["Subject", "Trial", "Event Type", "Code", "Time", "TTime",
                        "Uncertainty", "Duration", "Uncertainty", "ReqTime",
                        "ReqDur", "Stim Type", "Pair Index"]
    
    valid_line = re.compile(r'^(s|sub)[-_]?([1-9]|[1-4][0-9])[-_]se[1-5][^\t]*\t', re.IGNORECASE)

    code_index = None
    count = 0

    with open(log_path, "r") as f:
        for line in f:
            line = line.strip()

            if line.startswith("Subject\t"):
                columns = line.split("\t")
                if columns != expected_columns:
                    logging.error(f"❌ column mismatch in log file {subId}, run {run}")
                    raise ValueError(f"column mismatch in log file {subId}, run {run}")
                code_index = columns.index("Code")
                continue

            if valid_line.match(line):
                parts = line.split("\t")
                if len(parts) < len(expected_columns):
                    parts += [""] * (len(expected_columns) - len(parts))
                if parts[code_index] == "30":
                    count += 1

    if not count:
        logging.error(f"❌ frame count is 0 for {subId}, run {run}")
        raise ValueError(f"no frames found for {subId}, run {run}")
    else:
        logging.info(f"🟪 frame count = {count}")

    return count


In [5]:
def nifti_finder(task, subId, run):

    ###### input ######
    # task = "contact" (string "contact", "distance", "emotions" )
    #subId = "sub-01" (string sub-<two digit number>) 
    # run = 1 (int)

    logging.info(f"🔍 Searching for NIfTI for {subId}, run {run}")
    
    # HARDCODED_PATH
    nifti_folder_path = f"/scratch/nbe/prevent/Experiment1/Data_BIDS/derivatives/fmriprep/{subId}/func"

    run_str = f"run-{run:02}"

    matched_nifti_path = None
    for fname in os.listdir(nifti_folder_path):
        if (
            "2009" in fname
            and task in fname
            and fname.endswith("bold.nii.gz")
            and run_str in fname
        ):
            matched_nifti_path = os.path.join(nifti_folder_path, fname)
            logging.info(f"✅ Found NIfTI file: {fname}")
            break 

    else:
        logging.error(f"❌ No file found for {subId}, run {run}")
        raise ValueError(f"❌ No file found for {subId}, run {run}")
        
        
    return matched_nifti_path

In [12]:
task = "contact"
for sub in range(27, 43):
    subId = f"sub-{sub:02}"
    nifti_trimmer(task, subId)

2025-07-29 19:10:47,910 [INFO] ▶️ Trimming sub-27
2025-07-29 19:10:47,912 [INFO] 🔍 Searching for NIfTI for sub-27, run 1
2025-07-29 19:10:47,914 [INFO] ✅ Found NIfTI file: sub-27_task-contact_run-01_space-MNI152NLin2009cAsym_desc-preproc_bold.nii.gz
2025-07-29 19:10:47,915 [INFO] 🔍 Searching for NIfTI for sub-27, run 2
2025-07-29 19:10:47,915 [INFO] ✅ Found NIfTI file: sub-27_task-contact_run-02_space-MNI152NLin2009cAsym_desc-preproc_bold.nii.gz
2025-07-29 19:10:47,916 [INFO] 🔍 Searching for NIfTI for sub-27, run 3
2025-07-29 19:10:47,917 [INFO] ✅ Found NIfTI file: sub-27_task-contact_run-03_space-MNI152NLin2009cAsym_desc-preproc_bold.nii.gz
2025-07-29 19:10:47,918 [INFO] 🔍 Searching for NIfTI for sub-27, run 4
2025-07-29 19:10:47,919 [INFO] ✅ Found NIfTI file: sub-27_task-contact_run-04_space-MNI152NLin2009cAsym_desc-preproc_bold.nii.gz
2025-07-29 19:10:47,920 [INFO] 🔍 Searching for NIfTI for sub-27, run 5
2025-07-29 19:10:47,921 [INFO] ✅ Found NIfTI file: sub-27_task-contact_run-05_s